# Week 4 · PyTorch Tensors & Autograd: the engine under every neural net

# Requirements: pip install torch numpy
# Weeks 1-4 are CPU-only: no GPU and no API key are required.

This notebook builds the mental model for deep learning: **tensors** (data with a shape,
dtype, and device), **autograd** (the computation graph that fills in gradients for you),
and a tiny hand-built **MLP** trained end-to-end on toy data. It ends by printing the
MLP's final loss.

In [ ]:
import torch
import numpy as np

print("torch:", torch.__version__)

# Weeks 1-4 are CPU-only; detect an accelerator only to report it (we still train on CPU).
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

torch.manual_seed(0)   # determinism, always
np.random.seed(0)

### Tensors: data with a shape, a dtype, and a device

A tensor is a multi-dimensional array. A scalar is rank-0, a vector rank-1, a matrix
rank-2, a batch of matrices rank-3. In PyTorch a batch of `B` examples with `D` features
is a `(B, D)` tensor, and moving it to a GPU later is a single `.to(device)` call. The
shape discipline (tracking `(batch, features)` vs `(batch, classes)`) is most of
debugging.

In [ ]:
x = torch.randn(2, 3)
print("tensor:")
print(x)
print("shape:", tuple(x.shape), "| dtype:", x.dtype, "| device:", x.device)
print("mean:", round(x.mean().item(), 4), "| std:", round(x.std().item(), 4))
print()

print("scalar rank-0:", torch.tensor(3.14).dim())
print("vector rank-1:", torch.randn(4).dim())
print("matrix rank-2:", torch.randn(4, 5).dim())
print()

a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
print("matrix multiply a @ b:")
print(a @ b)

### Autograd: the computation graph

Set `requires_grad=True` and PyTorch records every operation into a **computation
graph**. After you compute a scalar loss, one call to `loss.backward()` walks that graph
in reverse and fills in `.grad` on every parameter. You write the forward math; the
framework derives the gradients, this is what "automatic differentiation" buys.

In [ ]:
# f(x) = x^2  ->  df/dx = 2x, so at x=3 the gradient is 6.
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print("f(x)=x^2; df/dx at x=3:", x.grad.item(), "(expected 6.0)")
assert abs(x.grad.item() - 6.0) < 1e-6

# Two variables flowing into a sum-of-squares loss: grad of sum(w^2) is 2w.
w = torch.tensor([2.0, -1.0], requires_grad=True)
loss = (w ** 2).sum()
loss.backward()
print("grad of sum(w^2) at w=[2,-1]:", w.grad.tolist(), "(expected [4.0, -2.0])")

### A tiny MLP, trained end-to-end

Now the whole engine in miniature: a `nn.Module` (a stack of `Linear` + `ReLU`), a
`MSELoss`, an optimizer, and the **training loop**, the same five lines for every model,
however fancy: `zero_grad → forward → loss → backward → step`. The toy data is exactly
linear (`y = 2*x0 - 1.5*x1 + 0.5*x2` plus tiny noise), so a near-zero final loss means
the net learned the signal.

In [ ]:
torch.manual_seed(0)
X = torch.randn(200, 3)
true_w = torch.tensor([2.0, -1.5, 0.5])
y = X @ true_w + 0.1 * torch.randn(200)

class TinyMLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(3, 16), torch.nn.ReLU(),
            torch.nn.Linear(16, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = TinyMLP()
loss_fn = torch.nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.05)

for epoch in range(400):
    opt.zero_grad()
    pred = model(X)
    loss = loss_fn(pred, y)
    loss.backward()
    opt.step()

print("final loss:", round(loss.item(), 6))
print("(near-zero loss = the MLP fit the linear signal plus its small noise.)")

### The metric

One number to close: the toy MLP's final training loss.

In [ ]:
print("TOY_MLP_FINAL_LOSS:", round(loss.item(), 6))